In [6]:
import pandas as pd
import numpy as np

# Đọc và xử lý dữ liệu
df = pd.read_csv('order_history_kaggle_data.csv')
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y', errors='coerce')
df['Hour'] = df['Order Placed At'].dt.hour
df['Day'] = df['Order Placed At'].dt.strftime('%a')  # Lấy ngày trong tuần dạng 'Sun', 'Mon', v.v.
df['Distance_km'] = df['Distance'].apply(lambda x: 0.5 if '<' in str(x) else float(str(x).replace('km', '')))
df = df[df['Order Status'] == 'Delivered']

# Đọc dữ liệu tốc độ từ SpeedPerHour.csv
speed_df = pd.read_csv('SpeedPerHour.csv', index_col=0)
speed_df.index = speed_df.index.str.replace('h', '').astype(int)  # Chuyển giờ thành số nguyên (0-23)
for col in speed_df.columns:
    speed_df[col] = speed_df[col].str.replace(' km/h', '').astype(float)  # Chuyển tốc độ thành số thực

In [7]:
# Tính thời gian giao hàng cho toàn bộ file CSV
def calculate_delivery_time(row):
    user_distance = row['Distance_km']
    user_hour = row['Hour']
    day = row['Day']
    user_time_in_minutes = row['Order Placed At'].hour * 60 + row['Order Placed At'].minute

    # Tra cứu tốc độ dựa trên giờ và ngày
    speed_at_hour_day = speed_df.loc[user_hour, day]

    # Tính thời gian giao hàng (phút) dựa trên khoảng cách và tốc độ
    delivery_time_minutes = (user_distance / speed_at_hour_day) * 60

    # Tính thời gian hoàn thành giao hàng
    end_time_minutes = user_time_in_minutes + delivery_time_minutes
    end_hour = int(end_time_minutes // 60) % 24  # Tính giờ kết thúc
    end_minute = end_time_minutes % 60  # Tính phút kết thúc

    return pd.Series({
        'Delivery Time (minutes)': delivery_time_minutes,
        'Speed (kmph)': speed_at_hour_day
    })

df[['Delivery Time (minutes)', 'Speed (kmph)']] = df.apply(calculate_delivery_time, axis=1, result_type='expand')

# In ra 20 dòng ngẫu nhiên để kiểm tra kết quả
print(df.sample(20)[['Order ID', 'Distance_km', 'Hour', 'Day', 'Delivery Time (minutes)', 'Speed (kmph)']])

         Order ID  Distance_km  Hour  Day  Delivery Time (minutes)  \
13661  6408570865          5.0    23  Tue                 8.823529   
6415   6260481136          7.0     0  Sat                11.666667   
17641  6498527700          2.0    12  Tue                 5.217391   
11268  6370143113          2.0    14  Tue                 5.217391   
2814   6224902671          6.0    20  Sun                13.333333   
20277  6552516144          4.0     2  Sun                 6.153846   
19961  6570877098          2.0    19  Fri                 5.714286   
3915   6203010410         10.0    23  Tue                17.647059   
2165   6175256701          1.0    13  Sat                 2.608696   
1099   6149463775          4.0    12  Thu                10.000000   
2632   6180395346          2.0    15  Sat                 5.217391   
17579  6513587880          7.0    13  Fri                17.500000   
5647   6276284524          0.5    19  Fri                 1.428571   
2396   6179324925   

In [8]:
# Đếm số lượng xuất hiện của các giá trị trong cột Distance_km và sắp xếp giảm dần
distance_counts = df['Distance_km'].value_counts().sort_values(ascending=False)

# In kết quả
print(distance_counts)

Distance_km
2.0     3538
1.0     3318
3.0     3185
4.0     2391
5.0     2118
6.0     2094
7.0     1267
9.0      757
8.0      706
0.5      642
10.0     378
11.0     248
12.0     112
16.0      85
14.0      82
15.0      76
13.0      62
17.0      26
18.0      23
19.0      18
20.0       3
21.0       2
Name: count, dtype: int64


Đa số đơn hàng tập trung ở khoảng cách (1-7 km), phản ánh nhu cầu đặt đồ ăn của khách hàng gần là chủ đạo. Các đơn hàng xa (trên 7 km) rất hiếm